# 02 — Chunk Filings and Upsert to Qdrant

This notebook processes the downloaded 10-K filings into section-aware chunks and indexes them in Qdrant for retrieval.

**Pipeline:**
1. Load `filing_metadata.csv` from the previous notebook
2. Detect 10-K section boundaries (Item 1, 1A, 7, 7A, 8, 9A; every other Item is labelled "Other") and split into labelled sections
3. Chunk each section into 800-token chunks with 100-token overlap
4. Save all chunks to `data/processed/filing_chunks.parquet`
5. Embed chunks with `text-embedding-3-small` and upsert to the `sec_filings` Qdrant collection

**Expected output:** ~3,000–8,000 chunks total across all 15 filings.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR, RAW_DIR
from src.chunking import process_all_filings, split_into_sections, chunk_section
from src.retrieval import get_qdrant_client, init_qdrant_collection, upsert_chunks

import os

metadata_path = os.path.join(PROCESSED_DIR, 'filing_metadata.csv')
metadata_df = pd.read_csv(metadata_path)
print(f"Loaded metadata for {len(metadata_df)} filings")
print(f"Available: {metadata_df['local_path'].notna().sum()} filings with local files")

## 2.1 — Section detection spot check

Before chunking everything, verify that section detection works correctly on one filing.

In [ ]:
sample_row = metadata_df[(metadata_df['ticker'] == 'MSFT') & (metadata_df['year'] == 2023)].iloc[0]
with open(sample_row['local_path'], 'r', encoding='utf-8') as f:
    sample_text = f.read()

sections = split_into_sections(sample_text)
print(f"MSFT 2023: detected {len(sections)} sections")
for s in sections:
    words = len(s['text'].split())
    print(f"  {s['section_name']:30s} {words:,} words")

## 2.2 — Process and chunk all filings

In [ ]:
chunks_path = os.path.join(PROCESSED_DIR, 'filing_chunks.parquet')

if os.path.exists(chunks_path):
    print(f"Loading existing chunks from {chunks_path}")
    chunks_df = pd.read_parquet(chunks_path)
else:
    print("Chunking all filings...")
    chunks_df = process_all_filings(metadata_df, raw_dir=RAW_DIR)

print(f"\nTotal chunks: {len(chunks_df):,}")
print(f"Columns: {list(chunks_df.columns)}")
chunks_df.head(3)

## 2.3 — Chunk statistics by company and section

In [ ]:
print("Chunks by company:")
print(chunks_df.groupby('ticker')['chunk_id'].count().sort_values(ascending=False))

print("\nChunks by section:")
print(chunks_df.groupby('section')['chunk_id'].count().sort_values(ascending=False))

print("\nAverage token count per chunk:")
print(chunks_df.groupby('section')['token_count'].mean().round(1).sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chunks by company × year heatmap
pivot = chunks_df.groupby(['ticker', 'year'])['chunk_id'].count().reset_index()
pivot = pivot.pivot(index='ticker', columns='year', values='chunk_id')
sns.heatmap(pivot, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Chunk Count by Company × Year')

# Token count distribution
axes[1].hist(chunks_df['token_count'], bins=40, edgecolor='black', color='steelblue')
axes[1].axvline(chunks_df['token_count'].mean(), color='red', linestyle='--', label=f"Mean: {chunks_df['token_count'].mean():.0f}")
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Chunk Token Count Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/chunk_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 — Initialize Qdrant and upsert chunks

In [ ]:
client = get_qdrant_client()
init_qdrant_collection(client, recreate=True)  # drop stale points from any previous chunking

# Check current collection size
from src.config import QDRANT_COLLECTION
collection_info = client.get_collection(QDRANT_COLLECTION)
print(f"Collection '{QDRANT_COLLECTION}'")
print(f"  Vectors indexed: {collection_info.vectors_count}")
print(f"  Points count:    {collection_info.points_count}")

In [ ]:
# Upsert all chunks. This will re-embed and overwrite if run again.
# Estimated cost: ~$0.02 for 6,000 chunks at text-embedding-3-small pricing
upsert_chunks(client, chunks_df)

# Verify final count
collection_info = client.get_collection(QDRANT_COLLECTION)
print(f"\nFinal collection size: {collection_info.points_count} points")

## 2.5 — Quick retrieval sanity check

In [ ]:
from src.retrieval import vanilla_retrieval, metadata_filtered_retrieval

test_query = "NVIDIA data center revenue growth"
results = vanilla_retrieval(client, test_query, top_k=5)

print(f"Query: '{test_query}'")
print(f"Top {len(results)} results:\n")
for r in results:
    print(f"  {r['ticker']} {r['year']} | {r['section']} | score={r['score']:.4f}")
    print(f"  {r['text'][:200]}...")
    print()